In [ ]:
import torch
from dotenv import load_dotenv
import yaml
from helper import load_default_dataloaders, load_ObjDet_model, load_best_model, train_model, plot_loss_curves

In [ ]:
torch.manual_seed(42)

In [ ]:
load_dotenv() 

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

## Loading Object Detection Model and Dataset

In [ ]:
dataset_path = "data/Traffic_Sign/car"
batch_size = 64
num_workers = 0 # multiple workers will not work in a ipynb on macOS

In [ ]:
# load class names from yaml file
with open(f"{dataset_path}/data.yaml", "r") as f:
    class_names = yaml.safe_load(f)["names"]

In [ ]:
# building the model with the new object detection head, getting the test dataloader from the weights preprocessing and the new input image size after the preprocessing transformations

model, test_transform, image_size = load_ObjDet_model(num_classes=len(class_names), frozen=True, device=device)

In [ ]:
train_dataloader, val_dataloader, _ = load_default_dataloaders(
    batch_size=batch_size,
    num_workers=num_workers, 
    image_size=image_size, 
    test_transform=test_transform, 
    dataset_path=dataset_path
)

## Defining Training and Loss Function Parameters

In [ ]:
# defining weights for the loss calculation
no_obj_w = 0.2 
loc_w = 4
obj_w = 12
class_w = 1
loc_l2_factor = 1

# defining training parameters
epochs = 20
learning_rate = 1e-5 # smaller learning rate to avoid destroying the learned features in the backbone 
weight_decay = 1e-4

## Fine-Tuning whole model with unfrozen backbone on Dataset

In [ ]:
# load best model from the model-head training phase
load_best_model(model)

# unfreeze all parameters in the model
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

wandb_config = {
    "epochs": epochs,
    "learning_rate": learning_rate,
    "batch_size": batch_size,
    "no_object_weight": no_obj_w,
    "loc_weight": loc_w,
    "objectness_weight": obj_w,
    "wh_l2_factor": loc_l2_factor,
    "optimizer": type(optimizer).__name__
}

train_losses, val_losses, best_val_loss = train_model(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    optimizer=optimizer,
    wandb_project_name="ObjDet-Full-Model-Finetune",
    wandb_config=wandb_config, 
    epochs=epochs
)

print("Model reached best validation loss of:", best_val_loss)


In [ ]:
plot_loss_curves(train_losses, val_losses)